# LeetCode 2079 — Watering Plants bằng Python

Link đề: https://leetcode.com/problems/watering-plants/ · Độ khó: **Medium** · Chủ đề: Array, Simulation

Skeleton LeetCode:

```python
class Solution:
    def wateringPlants(self, plants: List[int], capacity: int) -> int:
        ...
```

Lộ trình:
1. **Mô phỏng từng bước chân** (di chuyển từng đơn vị trên trục `x`) — đúng nghĩa đen của đề.
2. **Mô phỏng theo cây, cộng khoảng cách**: mỗi lần quay về sông trước cây `i` tốn thêm `2i` bước.
3. **Công thức đóng**: `n + Σ 2i` trên các cây phải nạp nước trước đó — viết một vòng `for` gọn.
4. **Phân rã theo chuyến đi** (trip): mỗi chuyến từ sông tới cây xa nhất rồi về — nhìn bài theo góc khác.

## 1. Mục tiêu học

1. Đọc kỹ luật mô phỏng: **tưới từ trái sang phải**, **không được nạp sớm**, chỉ nạp khi lượng nước còn lại **không đủ** cho cây kế tiếp.
2. Biết đổi mô phỏng "từng bước chân" (`O(n²)`) thành **tính khoảng cách** (`O(n)`): đi bộ là cộng hiệu tọa độ.
3. Tự suy ra công thức "mỗi lần nạp nước trước cây `i` tốn thêm `2i` bước" và kiểm chứng bằng mô phỏng.
4. Tránh các lỗi biên: so sánh `<` vs `<=`, tọa độ sông `x = -1`, nạp nước phải **đầy bình** (gán, không cộng).
5. Hiểu vì sao luật "không nạp sớm" là quan trọng: nếu được nạp sớm, tham lam không còn tối ưu (Bài 5).

## 2. Đề bài & ví dụ

Có `n` cây xếp hàng, cây `i` ở tọa độ `x = i`. Sông ở `x = -1` (nơi nạp nước). Bạn bắt đầu ở sông với bình **đầy** dung tích `capacity`. Cây `i` cần `plants[i]` đơn vị nước. Quy tắc:

- Tưới các cây **theo thứ tự từ trái sang phải**.
- Sau khi tưới cây hiện tại, nếu nước còn lại **không đủ để tưới trọn** cây tiếp theo thì quay về sông nạp **đầy** bình.
- **Không được** nạp nước sớm.

Mỗi bước di chuyển một đơn vị trên trục `x` tính là 1 bước. Trả về tổng số bước để tưới hết.

**Ví dụ 1:** `plants = [2,2,3,3]`, `capacity = 5` → `14` (`1 + 1 + 2 + 3 + 3 + 4`).

**Ví dụ 2:** `plants = [1,1,1,4,2,3]`, `capacity = 4` → `30`.

**Ví dụ 3:** `plants = [7,7,7,7,7,7,7]`, `capacity = 8` → `49` (phải nạp trước mỗi cây).

**Ràng buộc:** `1 ≤ n ≤ 1000`, `1 ≤ plants[i] ≤ 10⁶`, `max(plants) ≤ capacity ≤ 10⁹` (luôn tưới được mỗi cây bằng một bình đầy).

In [1]:
import random
import time
from typing import List

examples = [(([2, 2, 3, 3], 5), 14), (([1, 1, 1, 4, 2, 3], 4), 30), (([7, 7, 7, 7, 7, 7, 7], 8), 49)]
for (plants, cap), want in examples:
    print(f"plants = {plants}, capacity = {cap} -> kỳ vọng {want}")

plants = [2, 2, 3, 3], capacity = 5 -> kỳ vọng 14
plants = [1, 1, 1, 4, 2, 3], capacity = 4 -> kỳ vọng 30
plants = [7, 7, 7, 7, 7, 7, 7], capacity = 8 -> kỳ vọng 49


## 3. Kiến thức nền

### 3.1 Khoảng cách trên trục số

Đi từ `x = a` tới `x = b` tốn `|a - b|` bước. Sông ở `-1` ⇒ từ cây `j` về sông tốn `j + 1` bước, từ sông tới cây `i` tốn `i + 1` bước.

In [2]:
def dist(a, b):
    return abs(a - b)

RIVER = -1
print("cây 1 -> sông:", dist(1, RIVER), "| sông -> cây 2:", dist(RIVER, 2), "| cây 1 -> cây 2:", dist(1, 2))
print("đường vòng qua sông thay vì đi thẳng 1 -> 2 tốn thêm", dist(1, RIVER) + dist(RIVER, 2) - dist(1, 2), "= 2*2")

cây 1 -> sông: 2 | sông -> cây 2: 3 | cây 1 -> cây 2: 1
đường vòng qua sông thay vì đi thẳng 1 -> 2 tốn thêm 4 = 2*2


### 3.2 Biến trạng thái trong mô phỏng

Mô phỏng tốt cần chọn **đúng biến trạng thái**: ở đây là `pos` (vị trí hiện tại), `water` (nước còn trong bình), `steps` (tổng bước). Mỗi sự kiện (đi, tưới, nạp) cập nhật một vài biến.

In [3]:
state = {"pos": -1, "water": 5, "steps": 0}
def move_to(state, x):
    state["steps"] += dist(state["pos"], x); state["pos"] = x
move_to(state, 0); state["water"] -= 2
print(state)

{'pos': 0, 'water': 3, 'steps': 1}


### 3.3 Tổng cấp số cộng

Khi mọi cây đều phải nạp trước (ví dụ 3), số bước thêm là `Σ_{i=1}^{n-1} 2i = n(n-1)`. Tổng toàn bộ = `n + n(n-1) = n²`. Với `n = 7`: `49`. ✓

In [4]:
n = 7
print("n + sum(2i) =", n + sum(2 * i for i in range(1, n)), "| n^2 =", n * n)

n + sum(2i) = 49 | n^2 = 49


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 1 (`plants = [2,2,3,3]`, `capacity = 5`):

| bước | hành động | vị trí | nước | số bước cộng thêm | tổng |
|---|---|---|---|---|---|
| 1 | sông → cây 0, tưới 2 | 0 | 3 | 1 | 1 |
| 2 | cây 0 → cây 1, tưới 2 | 1 | 1 | 1 | 2 |
| 3 | còn 1 < 3 ⇒ về sông | -1 | 5 | 2 | 4 |
| 4 | sông → cây 2, tưới 3 | 2 | 2 | 3 | 7 |
| 5 | còn 2 < 3 ⇒ về sông | -1 | 5 | 3 | 10 |
| 6 | sông → cây 3, tưới 3 | 3 | 2 | 4 | 14 |

**Quan sát:** nếu không bao giờ phải nạp, ta chỉ đi `n = 4` bước (từ `-1` tới `3`). Mỗi lần nạp trước cây `i` biến "đi thẳng `i-1 → i` (1 bước)" thành "đi `i-1 → -1 → i`" (`i + (i + 1)` bước) ⇒ thêm `2i`. Ở đây nạp trước cây `2` và `3`: `4 + 2·2 + 2·3 = 14`. ✓

In [5]:
plants, cap = examples[0][0]
water, extra, refills = cap, 0, []
for i, need in enumerate(plants):
    if water < need:
        refills.append(i); extra += 2 * i; water = cap
    water -= need
print("nạp trước các cây:", refills, "-> n + extra =", len(plants), "+", extra, "=", len(plants) + extra)

nạp trước các cây: [2, 3] -> n + extra = 4 + 10 = 14


Hình dung quỹ đạo ví dụ 1 trên trục `x` (mỗi dòng là một đoạn đi):

```
x:   -1   0   1   2   3
     S -> 0 -> 1                 (2 bước, nước 5 -> 3 -> 1)
     S <------- 1                (2 bước về sông, nạp đầy 5)
     S ------------> 2           (3 bước, nước 5 -> 2)
     S <----------- 2            (3 bước về sông, nạp đầy 5)
     S ----------------> 3       (4 bước, nước 5 -> 2)
```

Tổng `2 + 2 + 3 + 3 + 4 = 14`. Đếm theo **chuyến**: chuyến 1 tới cây 1 và về (`2·2 = 4`), chuyến 2 tới cây 2 và về (`2·3 = 6`), chuyến cuối tới cây 3 (`4`) ⇒ `14` — chính là Cách 4.

## 5. Cách 1 — Brute Force / baseline: mô phỏng từng bước chân

**Ý tưởng:** đúng nghĩa đen — robot di chuyển **từng đơn vị** trên trục `x`, mỗi lần tăng `steps` thêm 1. Không cần suy luận gì về khoảng cách, nên dùng làm "trọng tài" để kiểm chứng các cách khác.

In [6]:
def water_unit_steps(plants, capacity):
    pos, water, steps = -1, capacity, 0
    for i, need in enumerate(plants):
        if water < need:                    # không đủ cho cây i ⇒ về sông (từng bước)
            while pos > -1:
                pos -= 1; steps += 1
            water = capacity
        while pos < i:                      # đi tới cây i (từng bước)
            pos += 1; steps += 1
        water -= need
    return steps

for (plants, cap), want in examples:
    print(water_unit_steps(plants, cap), "kỳ vọng", want)

14 kỳ vọng 14
30 kỳ vọng 30
49 kỳ vọng 49


**Độ phức tạp:** số lần lặp đúng bằng số bước = tối đa `n²` (khi nạp trước mọi cây) ⇒ `O(n²)` thời gian, `≈ 10⁶` với `n = 1000` — vẫn chạy được nhưng lãng phí. Space `O(1)`.

**Nút thắt:** đi từng bước một, trong khi khoảng cách tính được ngay bằng phép trừ tọa độ.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** tổng số bước = **(đường đi thẳng `-1 → n-1`, dài `n`)** + **(phụ phí của mỗi lần nạp)**. Phụ phí khi nạp ngay trước cây `i` (robot đang ở `i-1`) là

`(i - 1) - (-1)` (về sông) `+ i - (-1)` (từ sông tới cây `i`) `- 1` (đoạn thẳng `i-1 → i` lẽ ra đã đi) `= i + (i + 1) - 1 = 2i`.

**Quy tắc mô phỏng:**
- *Nếu* `water < plants[i]` *thì* nạp: `steps += 2i`, `water = capacity`.
- Tưới: `water -= plants[i]`.
- Kết quả `= n + steps`.

**Invariant:** trước khi xét cây `i`, `water` = lượng nước thực trong bình khi robot vừa tưới xong cây `i - 1` (hoặc `capacity` nếu `i = 0`).

Vì `capacity ≥ max(plants)`, sau khi nạp đầy luôn đủ tưới cây `i` ⇒ không bao giờ cần nạp hai lần liên tiếp.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mô phỏng theo cây, cộng khoảng cách

Vẫn giữ `pos` nhưng nhảy thẳng tới đích và cộng `|pos - đích|`. Dễ đọc, khớp từng dòng với mô tả của đề.

In [7]:
def water_jump_sim(plants, capacity):
    pos, water, steps = -1, capacity, 0
    for i, need in enumerate(plants):
        if water < need:
            steps += pos - (-1)              # về sông
            pos = -1
            water = capacity
        steps += i - pos                     # tới cây i
        pos = i
        water -= need
    return steps

for (plants, cap), want in examples:
    print(water_jump_sim(plants, cap), "kỳ vọng", want)

14 kỳ vọng 14
30 kỳ vọng 30
49 kỳ vọng 49


**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ.

### 7.2 Cách 3 — Công thức `n + Σ 2i` (code nộp)

Không cần `pos`: chỉ cần biết cây nào phải nạp trước.

In [8]:
def water_formula(plants, capacity):
    steps, water = 0, capacity
    for i, need in enumerate(plants):
        if water < need:
            steps += 2 * i
            water = capacity
        water -= need
    return steps + len(plants)

for (plants, cap), want in examples:
    print(water_formula(plants, cap), "kỳ vọng", want)

14 kỳ vọng 14
30 kỳ vọng 30
49 kỳ vọng 49


Một cách viết tương đương hay gặp: cộng `1` bước cho mỗi cây khi không nạp, và cộng `i + (i + 1) = 2i + 1` khi nạp:

```python
steps += 1 if water >= need else 2 * i + 1
```

Code nộp LeetCode của Cách 3: xem mục 9.

**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ.

### 7.3 Cách 4 — Phân rã theo chuyến đi

Một **chuyến** = rời sông với bình đầy, tưới một dãy cây liên tiếp `[s, e]`, rồi (nếu chưa xong) quay về sông. Chuyến không phải cuối dài `2(e + 1)` bước; chuyến cuối dài `e + 1 = n` bước (không quay về).

In [9]:
def trips(plants, capacity):
    res, s, water = [], 0, capacity
    for i, need in enumerate(plants):
        if water < need:
            res.append((s, i - 1)); s = i; water = capacity
        water -= need
    res.append((s, len(plants) - 1))
    return res

def water_by_trips(plants, capacity):
    ts = trips(plants, capacity)
    return sum(2 * (e + 1) for s, e in ts[:-1]) + len(plants)

for (plants, cap), want in examples:
    print(trips(plants, cap), "->", water_by_trips(plants, cap), "kỳ vọng", want)

[(0, 1), (2, 2), (3, 3)] -> 14 kỳ vọng 14
[(0, 2), (3, 3), (4, 4), (5, 5)] -> 30 kỳ vọng 30
[(0, 0), (1, 1), (2, 2), (3, 3), (4, 4), (5, 5), (6, 6)] -> 49 kỳ vọng 49


**Kiểm tra khớp với Cách 3:** chuyến `(s, e)` không phải cuối kết thúc trước cây `i = e + 1` — đúng là lần nạp trước cây `i`. `Σ 2(e + 1) = Σ 2i`. Và chuyến cuối dài `n`. Hai công thức trùng nhau — chỉ khác cách "chia bánh". ✓

Cách nhìn theo chuyến rất hữu ích cho các biến thể (Bài 5, mục 15).

## 8. TRACE từng bước

Trace Cách 2 trên ví dụ 2 (`plants = [1,1,1,4,2,3]`, `capacity = 4`): in `pos`, `water`, `steps` sau mỗi sự kiện.

In [10]:
def trace_sim(plants, capacity):
    pos, water, steps = -1, capacity, 0
    print(f"{'sự kiện':<26} | pos | water | steps")
    for i, need in enumerate(plants):
        if water < need:
            steps += pos + 1; pos = -1; water = capacity
            ev = f"về sông nạp (thiếu cho cây {i})"
            print(f"{ev:<26} | {pos:>3} | {water:>5} | {steps:>5}")
        steps += i - pos; pos = i; water -= need
        ev = f"tới cây {i}, tưới {need}"
        print(f"{ev:<26} | {pos:>3} | {water:>5} | {steps:>5}")
    return steps

trace_sim(*examples[1][0])

sự kiện                    | pos | water | steps
tới cây 0, tưới 1          |   0 |     3 |     1
tới cây 1, tưới 1          |   1 |     2 |     2
tới cây 2, tưới 1          |   2 |     1 |     3
về sông nạp (thiếu cho cây 3) |  -1 |     4 |     6
tới cây 3, tưới 4          |   3 |     0 |    10
về sông nạp (thiếu cho cây 4) |  -1 |     4 |    14
tới cây 4, tưới 2          |   4 |     2 |    19
về sông nạp (thiếu cho cây 5) |  -1 |     4 |    24
tới cây 5, tưới 3          |   5 |     1 |    30


30

In [11]:
def trace_formula(plants, capacity):
    steps, water = 0, capacity
    print(f"{'i':>2} | {'need':>4} | {'water trước':>11} | nạp? | {'+steps':>6} | {'steps':>5} | water sau")
    for i, need in enumerate(plants):
        before = water; add = 0
        if water < need:
            add = 2 * i; steps += add; water = capacity
        water -= need
        print(f"{i:>2} | {need:>4} | {before:>11} | {'có' if before < need else '':>4} | {add:>6} | {steps:>5} | {water}")
    print(f"=> {steps} + n({len(plants)}) = {steps + len(plants)}")

trace_formula(*examples[1][0])
trace_formula(*examples[2][0])

 i | need | water trước | nạp? | +steps | steps | water sau
 0 |    1 |           4 |      |      0 |     0 | 3
 1 |    1 |           3 |      |      0 |     0 | 2
 2 |    1 |           2 |      |      0 |     0 | 1
 3 |    4 |           1 |   có |      6 |     6 | 0
 4 |    2 |           0 |   có |      8 |    14 | 2
 5 |    3 |           2 |   có |     10 |    24 | 1
=> 24 + n(6) = 30
 i | need | water trước | nạp? | +steps | steps | water sau
 0 |    7 |           8 |      |      0 |     0 | 1
 1 |    7 |           1 |   có |      2 |     2 | 1
 2 |    7 |           1 |   có |      4 |     6 | 1
 3 |    7 |           1 |   có |      6 |    12 | 1
 4 |    7 |           1 |   có |      8 |    20 | 1
 5 |    7 |           1 |   có |     10 |    30 | 1
 6 |    7 |           1 |   có |     12 |    42 | 1
=> 42 + n(7) = 49


Đọc bảng ví dụ 2: nạp trước cây `3`, `4`, `5` ⇒ phụ phí `6 + 8 + 10 = 24`, cộng `n = 6` ⇒ `30`. ✓

Ví dụ 3: nạp trước mọi cây `1..6` ⇒ `2(1 + … + 6) = 42`, cộng `7` ⇒ `49 = 7²`. ✓

## 9. Code nộp LeetCode

In [12]:
class Solution:
    def wateringPlants(self, plants: List[int], capacity: int) -> int:
        steps = 0
        water = capacity
        for i, need in enumerate(plants):
            if water < need:          # không đủ ⇒ phải về sông trước cây i
                steps += 2 * i        # đường vòng i bước về + (i + 1) bước tới, trừ 1 bước đi thẳng
                water = capacity      # nạp ĐẦY (gán, không cộng)
            water -= need
        return steps + len(plants)    # n bước đi thẳng từ sông tới cây cuối

sol = Solution()
for (plants, cap), want in examples:
    got = sol.wateringPlants(plants, cap)
    print(got, "OK" if got == want else "SAI")

14 OK
30 OK
49 OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (quỹ đạo).** Theo luật đề, quỹ đạo của robot là: đi từ sông sang phải, mỗi khi gặp "điểm nạp" trước cây `i` thì đi `i - 1 → -1 → i`. Luật "không nạp sớm" và "tưới trái sang phải" làm quỹ đạo **xác định duy nhất** — không có lựa chọn nào. Vì vậy bài là thuần mô phỏng, không cần tối ưu.

**Bổ đề 2 (tập điểm nạp).** Robot nạp trước cây `i` ⇔ lượng nước còn sau khi tưới cây `i - 1` nhỏ hơn `plants[i]`. Biến `water` trong code đúng bằng lượng nước đó (invariant ở mục 6): khởi tạo `capacity`, mỗi lần nạp gán lại `capacity`, mỗi lần tưới trừ `plants[i]`. Vì `capacity ≥ plants[i]`, sau khi nạp `water - plants[i] ≥ 0` — không bao giờ âm.

**Bổ đề 3 (độ dài).** Quỹ đạo không nạp lần nào có độ dài `n`. Thay đoạn `i - 1 → i` (1 bước) bằng `i - 1 → -1 → i` (`i + i + 1` bước) tăng độ dài đúng `2i`. Các lần thay thế ở các `i` khác nhau tác động lên các đoạn rời nhau ⇒ cộng dồn được. Vậy tổng `= n + Σ_{i ∈ R} 2i`. ✓

**Cách 4** chia cùng quỹ đạo thành các chuyến; tổng độ dài các chuyến bằng độ dài quỹ đạo.

## 11. Phân tích Time / Space

Gọi `n = len(plants) ≤ 1000`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Từng bước chân | `O(n²)` (= số bước) | `O(1)` | trọng tài |
| 2. Nhảy + cộng khoảng cách | `O(n)` | `O(1)` | sát mô tả đề |
| 3. Công thức `n + Σ 2i` | `O(n)` | `O(1)` | **nộp bài** |
| 4. Theo chuyến | `O(n)` | `O(số chuyến)` | hữu ích cho biến thể |

Số bước tối đa: `n² = 10⁶` (mỗi cây cần nạp) — vừa `int` thường. Đo thực tế:

In [13]:
rng = random.Random(1)
worst = ([10**6] * 1000, 10**6)                                   # nạp trước mọi cây
rand = ([rng.randint(1, 10**6) for _ in range(1000)], 2 * 10**6)
for name, fn in [("unit steps", water_unit_steps), ("jump sim", water_jump_sim), ("formula", water_formula),
                 ("trips", water_by_trips), ("submit", Solution().wateringPlants)]:
    t0 = time.perf_counter(); a = fn(*worst); b = fn(*rand)
    print(f"{name:10s}: worst = {a}, random = {b}  ({(time.perf_counter() - t0) * 1000:.1f} ms)")

unit steps: worst = 1000000, random = 300792  (35.1 ms)
jump sim  : worst = 1000000, random = 300792  (0.1 ms)
formula   : worst = 1000000, random = 300792  (0.1 ms)
trips     : worst = 1000000, random = 300792  (0.2 ms)
submit    : worst = 1000000, random = 300792  (0.1 ms)


## 12. Bộ test

In [14]:
fns = [water_jump_sim, water_formula, water_by_trips, lambda p, c: Solution().wateringPlants(p, c)]

for (plants, cap), want in examples:
    assert water_unit_steps(plants, cap) == want
    for fn in fns:
        assert fn(plants, cap) == want

edge = [
    (([1], 1), 1),                      # một cây
    (([5], 10), 1),
    (([3, 2], 5), 2),                   # vừa khít: 3 + 2 = 5, không nạp
    (([3, 3], 5), 4),                   # nạp trước cây 1: 2 + 2*1
    (([1, 1, 1, 1], 10**9), 4),         # không bao giờ nạp
    (([2, 2, 2, 2], 2), 16),            # nạp trước mọi cây: n^2
    (([1, 2, 1, 2], 3), 4 + 2 * 2),     # 1+2=3 vừa khít, nạp trước cây 2; 1+2 = 3 vừa khít
]
for (plants, cap), want in edge:
    assert water_unit_steps(plants, cap) == want, (plants, cap)
    for fn in fns:
        assert fn(plants, cap) == want, (fn, plants, cap)

rng = random.Random(2024)
for _ in range(1000):
    n = rng.randint(1, 30)
    plants = [rng.randint(1, 10) for _ in range(n)]
    cap = rng.randint(max(plants), max(plants) + rng.choice([0, 3, 10, 100]))
    want = water_unit_steps(plants, cap)
    for fn in fns:
        assert fn(plants, cap) == want, (fn, plants, cap)
    assert want == n * n or want < n * n           # chặn trên n^2
print("ví dụ + edge + 1000 bộ ngẫu nhiên khớp mô phỏng từng bước")

ví dụ + edge + 1000 bộ ngẫu nhiên khớp mô phỏng từng bước


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng `<=` thay vì `<` (nạp khi vừa đủ)

In [15]:
def wrong_le(plants, capacity):
    steps, water = 0, capacity
    for i, need in enumerate(plants):
        if water <= need and i > 0:             # SAI: vừa đủ vẫn được tưới, không phải nạp
            steps += 2 * i; water = capacity
        water -= need
    return steps + len(plants)

for (plants, cap), want in examples + [(([3, 2], 5), 2)]:
    print(f"{plants}, cap={cap}: sai = {wrong_le(plants, cap)} | đúng = {want}")

[2, 2, 3, 3], cap=5: sai = 14 | đúng = 14
[1, 1, 1, 4, 2, 3], cap=4: sai = 30 | đúng = 30
[7, 7, 7, 7, 7, 7, 7], cap=8: sai = 49 | đúng = 49
[3, 2], cap=5: sai = 4 | đúng = 2


Đề: nạp khi **không đủ để tưới trọn** cây tiếp theo ⇒ `water < need`. Với `[3, 2]`, `capacity = 5`: sau cây 0 còn `2`, vừa đủ cây 1 — không nạp. **Sửa:** `<`.

### Lỗi 2 — Sai phụ phí: `2 * (i + 1)` hoặc `i + 1`

In [16]:
def wrong_extra_2i_plus_2(plants, capacity):
    steps, water = 0, capacity
    for i, need in enumerate(plants):
        if water < need:
            steps += 2 * (i + 1); water = capacity      # SAI: tính cả quãng tới cây i như thể từ cây i quay về
        water -= need
    return steps + len(plants)

def wrong_extra_no_return(plants, capacity):
    steps, water = 0, capacity
    for i, need in enumerate(plants):
        if water < need:
            steps += i + 1; water = capacity            # SAI: quên quãng quay về sông
        water -= need
    return steps + len(plants)

for (plants, cap), want in examples:
    print(f"sai(2(i+1)) = {wrong_extra_2i_plus_2(plants, cap)} | sai(i+1) = {wrong_extra_no_return(plants, cap)} | đúng = {want}")

sai(2(i+1)) = 18 | sai(i+1) = 11 | đúng = 14
sai(2(i+1)) = 36 | sai(i+1) = 21 | đúng = 30
sai(2(i+1)) = 61 | sai(i+1) = 34 | đúng = 49


Robot nạp khi đang đứng ở cây `i - 1` (chưa tới cây `i`): về sông `i` bước, tới cây `i` thêm `i + 1` bước, thay cho 1 bước đi thẳng ⇒ phụ phí `2i`. **Sửa:** tính lại cẩn thận bằng tọa độ, hoặc dùng Cách 2 với `pos` rõ ràng.

### Lỗi 3 — Nạp bằng `water += capacity` (tràn bình)

In [17]:
def wrong_add_capacity(plants, capacity):
    steps, water = 0, capacity
    for i, need in enumerate(plants):
        if water < need:
            steps += 2 * i; water += capacity           # SAI: bình chỉ chứa tối đa capacity
        water -= need
    return steps + len(plants)

rng = random.Random(3)
for _ in range(10**4):
    plants = [rng.randint(1, 6) for _ in range(rng.randint(2, 6))]
    cap = rng.randint(max(plants), 8)
    if wrong_add_capacity(plants, cap) != water_formula(plants, cap):
        print("phản ví dụ:", plants, "cap =", cap, "| sai =", wrong_add_capacity(plants, cap), "| đúng =", water_formula(plants, cap))
        break

phản ví dụ: [4, 6, 5, 1, 5, 1] cap = 7 | sai = 22 | đúng = 20


Nước thừa từ chuyến trước bị "cộng dồn" ⇒ nạp ít lần hơn thực tế. **Sửa:** `water = capacity`.

### Lỗi 4 — Bắt đầu ở `x = 0` thay vì `x = -1`

In [18]:
def wrong_start_zero(plants, capacity):
    pos, water, steps = 0, capacity, 0                  # SAI: sông ở -1
    for i, need in enumerate(plants):
        if water < need:
            steps += pos + 1; pos = -1; water = capacity
        steps += i - pos; pos = i; water -= need
    return steps

for (plants, cap), want in examples:
    print("sai =", wrong_start_zero(plants, cap), "| đúng =", want)

sai = 13 | đúng = 14
sai = 29 | đúng = 30
sai = 48 | đúng = 49


Thiếu đúng 1 bước đầu tiên (sông → cây 0). **Sửa:** `pos = -1`.

### Lỗi 5 — Kiểm tra nạp **sau** khi đã trừ nước của cây hiện tại cho chính cây đó

Viết `water -= need` rồi mới `if water < 0: nạp` — khi đó đã "tưới" bằng nước không có. Kết quả bước có thể vẫn đúng nếu xử lý khéo, nhưng rất dễ quên hoàn lại nước. Cấu trúc an toàn: **kiểm tra trước, tưới sau**.

In [19]:
def wrong_check_after(plants, capacity):
    steps, water = 0, capacity
    for i, need in enumerate(plants):
        water -= need
        if water < 0:
            steps += 2 * i
            water = capacity                            # SAI: quên trừ need của cây i sau khi nạp
    return steps + len(plants)

for (plants, cap), want in examples:
    print("sai =", wrong_check_after(plants, cap), "| đúng =", want)

sai = 8 | đúng = 14
sai = 22 | đúng = 30
sai = 25 | đúng = 49


**Sửa:** sau khi nạp phải `water = capacity - need` (hoặc dùng cấu trúc kiểm tra-trước).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề "mô phỏng có quy tắc cứng, không có lựa chọn" ⇒ chỉ cần mô phỏng chính xác; tìm cách **tính khoảng cách** thay vì đi từng bước.
- Tách chi phí thành **phần cố định** (đi thẳng `n` bước) + **phụ phí theo sự kiện** (`2i` mỗi lần nạp) — kỹ thuật "chi phí cơ sở + phụ phí" dùng được ở nhiều bài.
- Luôn tự hỏi: so sánh là `<` hay `<=`? Đọc từ "không đủ" ⇒ `<`.
- Trạng thái "đầy lại" là **gán**, không phải cộng.
- Họ bài: 2105 *Watering Plants II* (hai người tưới từ hai đầu), 2073 *Time Needed to Buy Tickets*, 1094 *Car Pooling*.

## 15. Biến thể của bài

1. **2105 Watering Plants II:** Alice tưới từ trái, Bob từ phải, cùng lúc; đếm số lần nạp. (Bài kế tiếp trong danh sách.)
2. **Được nạp sớm** (bỏ luật "không nạp sớm"): bài trở thành **tối ưu** — tham lam không còn đúng, cần DP (Bài 5).
3. **Sông ở cả hai đầu** (`x = -1` và `x = n`): khi nạp chọn sông gần hơn.
4. **Số lần nạp** thay vì số bước: chỉ đếm `R`.
5. **Mọi cây cùng lượng nước `p`**: mỗi chuyến tưới `k = capacity // p` cây ⇒ có công thức đóng.

In [20]:
def refills_count(plants, capacity):
    """Biến thể 4."""
    water, r = capacity, 0
    for need in plants:
        if water < need:
            r += 1; water = capacity
        water -= need
    return r

def equal_plants_closed(n, p, capacity):
    """Biến thể 5: nạp trước các cây k, 2k, 3k, ... (k = capacity // p)."""
    k = capacity // p
    m = (n - 1) // k                               # số lần nạp
    return n + 2 * k * m * (m + 1) // 2

def two_rivers(plants, capacity):
    """Biến thể 3: nạp ở sông gần hơn (x = -1 hoặc x = n), mô phỏng với pos."""
    n = len(plants); pos, water, steps = -1, capacity, 0
    for i, need in enumerate(plants):
        if water < need:
            left = (pos + 1) + (i + 1)            # về -1 rồi tới i
            right = (n - pos) + (n - i)           # tới n rồi quay lại i
            steps += min(left, right); pos = i; water = capacity - need
            continue
        steps += i - pos; pos = i; water -= need
    return steps

for n in range(1, 30):
    for p in range(1, 5):
        for cap in range(p, 12):
            assert equal_plants_closed(n, p, cap) == water_formula([p] * n, cap)
print("số lần nạp ví dụ:", [refills_count(p, c) for (p, c), _ in examples])
print("công thức đóng (cây bằng nhau) khớp mô phỏng; ví dụ 3:", equal_plants_closed(7, 7, 8))
print("hai sông, ví dụ:", [two_rivers(p, c) for (p, c), _ in examples], "(biến thể tự đặt)")

số lần nạp ví dụ: [2, 3, 6]
công thức đóng (cây bằng nhau) khớp mô phỏng; ví dụ 3: 49
hai sông, ví dụ: [10, 18, 31] (biến thể tự đặt)


In [21]:
# Kiểm chứng biến thể 3 (hai sông) bằng mô phỏng từng bước chân
def two_rivers_unit(plants, capacity):
    n = len(plants); pos, water, steps = -1, capacity, 0
    for i, need in enumerate(plants):
        if water < need:
            go_left = (pos + 1) + (i + 1) <= (n - pos) + (n - i)
            target = -1 if go_left else n
            while pos != target:
                pos += 1 if target > pos else -1; steps += 1
            water = capacity
        while pos != i:
            pos += 1 if i > pos else -1; steps += 1
        water -= need
    return steps

rng = random.Random(15)
for _ in range(500):
    plants = [rng.randint(1, 6) for _ in range(rng.randint(1, 12))]
    cap = rng.randint(max(plants), 9)
    assert two_rivers(plants, cap) == two_rivers_unit(plants, cap) <= water_formula(plants, cap)
print("hai sông: công thức khớp mô phỏng từng bước; luôn không tệ hơn một sông")

hai sông: công thức khớp mô phỏng từng bước; luôn không tệ hơn một sông


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Từng bước chân | nghĩa đen | `O(n²)` | `O(1)` | kiểm chứng |
| 2. Nhảy + khoảng cách | giữ `pos` | `O(n)` | `O(1)` | khi quỹ đạo phức tạp (nhiều sông…) |
| 3. `n + Σ 2i` | cơ sở + phụ phí | `O(n)` | `O(1)` | **nộp bài** |
| 4. Theo chuyến | chia quỹ đạo | `O(n)` | `O(#chuyến)` | biến thể tối ưu hóa |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `plants = [3, 1, 3]`, `capacity = 4`? Và `plants = [1, 2, 3, 4]`, `capacity = 10`? (Ví dụ tự đặt.)

**Bài 2 (tính tay):** `plants = [2, 4, 5, 1, 2]`, `capacity = 6`. Liệt kê các cây phải nạp trước, các chuyến đi và tổng số bước.

**Bài 3 (điền TODO):** hoàn thiện `refill_points(plants, capacity)` trả về danh sách chỉ số `i` mà robot phải nạp ngay trước cây `i`.

**Bài 4 (tự viết lại):** viết `steps_after_k_plants(plants, capacity, k)` = số bước tới lúc vừa tưới xong cây `k - 1` (robot đứng ở cây `k - 1`). Kiểm chứng với `k = n` cho ra đáp án bài gốc.

**Bài 5 (mini challenge):** **được phép nạp sớm** (bất cứ lúc nào, nhưng vẫn tưới trái sang phải và phải tưới trọn mỗi cây trong một lần). Tìm số bước **nhỏ nhất**. Chứng tỏ tham lam (chỉ nạp khi thiếu) có thể không tối ưu và viết DP `O(n²)`. (Biến thể tự đặt — đối chiếu lại đề gốc nếu dùng làm đề.)

In [22]:
def refill_points(plants, capacity):
    # TODO: mô phỏng như Cách 3, thu các i có water < plants[i]
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) `[3,1,3]`: sau cây 0 còn 1, đủ cây 1, còn 0 < 3 ⇒ nạp trước cây 2. (2) Phụ phí `4`, `n = 3`. (3) Ví dụ thứ hai không bao giờ nạp ⇒ `4`.

**Bài 2:** (1) Còn `4` sau cây 0, đủ cây 1 ⇒ còn `0`. (2) Nạp trước cây 2 (còn `1` sau tưới), đủ cây 3 ⇒ còn `0`, nạp trước cây 4. (3) `5 + 2·2 + 2·4 = 17`.

**Bài 3:** (1) Như vòng lặp Cách 3. (2) `res.append(i)` khi nạp. (3) Kiểm tra `n + 2·sum(res)` bằng đáp án.

**Bài 4:** (1) Chạy Cách 3 trên `plants[:k]`. (2) Robot đứng ở `k - 1` ⇒ phần cơ sở là `k`. (3) Hoặc chạy Cách 2 dừng sau cây `k - 1`.

**Bài 5:** (1) Tổng bước `= n + Σ_{i ∈ R} 2i` vẫn đúng với **mọi** tập điểm nạp `R` hợp lệ. (2) `R` hợp lệ ⇔ mỗi đoạn giữa hai điểm nạp có tổng `≤ capacity`. (3) `dp[j]` = chi phí nhỏ nhất khi có một lần nạp ngay trước cây `j` và mọi cây `< j` đã tưới; `dp[j] = 2j + min dp[i]` với `sum(plants[i:j]) ≤ capacity`; đáp án `n + min dp[i]` với `sum(plants[i:]) ≤ capacity`.

## 19. Đáp án tham khảo

In [23]:
# Bài 1
assert Solution().wateringPlants([3, 1, 3], 4) == water_unit_steps([3, 1, 3], 4) == 7
assert Solution().wateringPlants([1, 2, 3, 4], 10) == 4

# Bài 2
p2, c2 = [2, 4, 5, 1, 2], 6
print("Bài 2: chuyến =", trips(p2, c2), "| số bước =", water_formula(p2, c2))
assert water_unit_steps(p2, c2) == 17

# Bài 3
def refill_points_ans(plants, capacity):
    res, water = [], capacity
    for i, need in enumerate(plants):
        if water < need:
            res.append(i); water = capacity
        water -= need
    return res
for (plants, cap), want in examples:
    R = refill_points_ans(plants, cap)
    assert len(plants) + 2 * sum(R) == want
    print("điểm nạp:", R)

# Bài 4
def steps_after_k_plants_ans(plants, capacity, k):
    return water_formula(plants[:k], capacity)
rng = random.Random(44)
for _ in range(300):
    plants = [rng.randint(1, 9) for _ in range(rng.randint(1, 12))]
    cap = rng.randint(max(plants), 15)
    assert steps_after_k_plants_ans(plants, cap, len(plants)) == water_unit_steps(plants, cap)
    k = rng.randint(1, len(plants))
    assert steps_after_k_plants_ans(plants, cap, k) == water_unit_steps(plants[:k], cap)
print("Bài 1-4: OK")

Bài 2: chuyến = [(0, 1), (2, 3), (4, 4)] | số bước = 17
điểm nạp: [2, 3]
điểm nạp: [3, 4, 5]
điểm nạp: [1, 2, 3, 4, 5, 6]
Bài 1-4: OK


In [24]:
# Bài 5 — được nạp sớm: DP trên tập điểm nạp
from itertools import combinations

def early_refill_dp(plants, capacity):
    n = len(plants)
    pre = [0]
    for x in plants: pre.append(pre[-1] + x)
    INF = float("inf")
    dp = [INF] * (n + 1)          # dp[j]: chi phí phụ phí nhỏ nhất, chuyến hiện tại bắt đầu ở cây j
    dp[0] = 0
    for j in range(1, n):
        for i in range(j):
            if pre[j] - pre[i] <= capacity and dp[i] < INF:
                dp[j] = min(dp[j], dp[i] + 2 * j)
    return n + min(dp[i] for i in range(n) if pre[n] - pre[i] <= capacity)

def early_refill_brute(plants, capacity):
    n = len(plants); best = None
    for r in range(n):
        for R in combinations(range(1, n), r):
            starts = [0] + list(R) + [n]
            if all(sum(plants[starts[t]:starts[t + 1]]) <= capacity for t in range(len(starts) - 1)):
                cost = n + 2 * sum(R)
                best = cost if best is None else min(best, cost)
    return best

rng = random.Random(55)
gap = None
for _ in range(1500):
    plants = [rng.randint(1, 5) for _ in range(rng.randint(1, 8))]
    cap = rng.randint(max(plants), 9)
    d = early_refill_dp(plants, cap)
    assert d == early_refill_brute(plants, cap)
    assert d <= water_formula(plants, cap)               # nạp sớm không bao giờ tệ hơn
    if gap is None and d < water_formula(plants, cap):
        gap = (plants, cap, water_formula(plants, cap), d)
print("DP khớp brute force; tham lam không tối ưu khi được nạp sớm, ví dụ:", gap)
print("[3,1,3], cap 4: tham lam =", water_formula([3, 1, 3], 4), "| nạp sớm trước cây 1 =", early_refill_dp([3, 1, 3], 4))

DP khớp brute force; tham lam không tối ưu khi được nạp sớm, ví dụ: ([5, 2, 2, 4, 2, 2], 6, 24, 20)
[3,1,3], cap 4: tham lam = 7 | nạp sớm trước cây 1 = 5


Ở `[3, 1, 3]`, `capacity = 4`: tham lam tưới cây 0, 1 (tổng 4) rồi nạp trước cây 2 (phụ phí 4). Nạp **sớm** trước cây 1 (phụ phí 2) rồi tưới cây 1, 2 (tổng 4) chỉ tốn `3 + 2 = 5` bước. Luật "không nạp sớm" của đề chính là thứ biến bài tối ưu thành bài mô phỏng.

## 20. Template nhớ nhanh + Checklist

```python
# Mô phỏng "đi từ sông, nạp khi thiếu": chi phí = phần cơ sở + phụ phí mỗi sự kiện
steps, water = 0, capacity
for i, need in enumerate(plants):
    if water < need:          # KHÔNG đủ (strict)
        steps += 2 * i        # (i về sông) + (i + 1 tới cây i) - 1 bước đi thẳng
        water = capacity      # gán đầy
    water -= need
return steps + len(plants)
```

**Checklist trước khi submit:**
- [ ] So sánh `water < need` (vừa đủ thì không nạp).
- [ ] Phụ phí `2i` (robot đang ở `i - 1`), không phải `2(i + 1)`.
- [ ] Nạp là **gán** `water = capacity`.
- [ ] Cộng phần cơ sở `n` (sông ở `-1` → cây `n - 1`).
- [ ] Kiểm tra trước, tưới sau.